<div style="background-color:#000047; padding:30px; border-radius:10px; color:white; text-align:center;">
    <img src='Figures/alinco_white_text.png' style="height:100px; margin-bottom:10px;"/>
    <h1>Módulo 3: Modelos de Lenguaje</h1>
    <h2>N-gramas Profundos (Deep N-grams)</h2>
</div>

En este archivo de jupyter exploraremos las Redes Neuronales Recurrentes `RNN`.
- Usaremos los fundamentos del paquete [trax](https://github.com/google/trax) de google para implementar cualquier tipo de modelo de aprendizaje profundo. 


Lo que haremos será predecir el siguiente conjunto de caracteres usando los caracteres anteriores. 
- Aunque esta tarea suena simple, es bastante útil.
- Comenzaremos convirtiendo una línea de texto en un tensor
- Luego crearemos un generador para alimentar datos al modelo
- Entrenaremos una red neuronal para predecir el nuevo conjunto de caracteres de longitud definida. 
- Usaremos embeddings para cada carácter y los alimentaremos como entradas a nuestro modelo. 
- El modelo convertirá cada carácter a su embedding, pasará los embeddings a través de una Unidad Recurrente con Compuertas `GRU`, y los pasará por una capa lineal para predecir el siguiente conjunto de caracteres.

<img src = "Figures/model.png" style="width:600px;height:150px;"/>

La figura de arriba nos da un resumen de lo que implementaremos. 
- Obtendremos los embeddings
- Apilaremos los embeddings uno encima de otro
- Los Pasaremos por dos capas con una activación relu en el medio
- Finalmente, Calcularemos los datos de salida apartir de la función de activación softmax. 

Para predecir el siguiente carácter:
- Usaremos la salida softmax e identificaremos la palabra con la mayor probabilidad.
- La palabra con la mayor probabilidad es la predicción para la siguiente palabra.

In [ ]:
import os
import trax
import trax.fastmath.numpy as np
import pickle
import numpy as np
import random as rnd
from trax import fastmath
from trax import layers as tl

# establecer la semilla aleatoria
rnd.seed(31)
np.random.seed(31)

# Importando los Datos

### Cargando los datos

<img src = "Figures/shakespeare.png" style="width:250px;height:250px;"/>

Ahora importaremos el conjunto de datos y realizaremos algo de procesamiento. 
- El conjunto de datos tiene una oración por línea.
- Estaremos generando caracteres, así que tenemos que procesar cada oración convirtiendo cada **carácter** (y no palabra) a un número. 
- Usaremos la función `ord` para convertir un carácter único a un ID entero único. 
- Almacenaremos cada línea en una lista.
- Crearemos un generador de datos que reciba el `batch_size` y el `max_length`. 
    - El `max_length` corresponde a la longitud máxima de la oración.

In [ ]:
dirname = 'Data/data_sqm/'
lines = [] # almacenando todas las lineas en una variable. 
for filename in os.listdir(dirname):
    # solo procesar archivos de texto (ignorar .gz, .pkl, etc.)
    if not filename.endswith('.txt'):
        continue
    with open(os.path.join(dirname, filename), encoding='utf-8') as files:
        for line in files:
            # eliminar los espacios en blanco al inicio y al final
            pure_line = line.strip()
            
            # si pure_line no es la cadena vacia,
            if pure_line:
                # agregarla a la lista
                lines.append(pure_line)

In [ ]:
n_lines = len(lines)
print(f"Number of lines: {n_lines}")
print(f"Sample line at position 0 {lines[0]}")
print(f"Sample line at position 999 {lines[999]}")

Observemos que las letras están tanto en mayúsculas como en minúsculas. Para reducir la complejidad del procesamiento, convertiremos todos los caracteres a minúsculas. De esta manera, el modelo solo necesita predecir la probabilidad de que una letra sea 'a' y no decidir entre la 'A' mayúscula y la 'a' minúscula.

In [ ]:
# recorrer cada linea
for i, line in enumerate(lines):
    # convertir todo a minusculas
    lines[i] = line.lower()

print(f"Number of lines: {n_lines}")
print(f"Sample line at position 0 {lines[0]}")
print(f"Sample line at position 999 {lines[999]}")

In [ ]:
eval_lines = lines[-1000:] # Crear un conjunto de validacion reservado (holdout)
lines = lines[:-1000] # Dejar el resto para entrenamiento

print(f"Number of lines for training: {len(lines)}")
print(f"Number of lines for validation: {len(eval_lines)}")

### Convertir una línea a tensor

Ahora que tenemos una lista de líneas, convertiremos cada carácter de esa lista a un número. Podemos usar la función `ord` de Python para hacerlo. 

Dada una cadena que representa un carácter Unicode, la función `ord` devuelve un entero que representa el punto de código Unicode de ese carácter.

In [ ]:
# Ver el entero unicode unico asociado a cada caracter
print(f"ord('a'): {ord('a')}")
print(f"ord('b'): {ord('b')}")
print(f"ord('c'): {ord('c')}")
print(f"ord(' '): {ord(' ')}")
print(f"ord('x'): {ord('x')}")
print(f"ord('y'): {ord('y')}")
print(f"ord('z'): {ord('z')}")
print(f"ord('1'): {ord('1')}")
print(f"ord('2'): {ord('2')}")
print(f"ord('3'): {ord('3')}")

Ahora implementaremos una función que reciba una sola línea y transforme cada carácter en su entero unicode. Esto devuelve una lista de enteros, a la que nos referiremos como un tensor.
- Usaremos un entero especial para representar el fin de la oración (el fin de la línea).
- Este será el parámetro EOS_int (entero de fin de oración) de la función.
- Incluiremos el EOS_int como el último entero de la 
- Usaremos el número `1` para representar el fin de una oración.

In [ ]:
def line_to_tensor(line, EOS_int=1):    
    # Inicializar el tensor como una lista vacia
    tensor = []
    ### START CODE HERE (Replace instances of 'None' with your code) ###
    # para cada caracter:
    for c in line:
        
        # convertir a entero unicode
        c_int = ord(c)
        
        # agregar el entero unicode a la lista tensor
        tensor.append(c_int)
    
    # incluir el entero de fin de oracion
    tensor.append(EOS_int)
    ### END CODE HERE ###

    return tensor

In [ ]:
# Probando tu salida
line_to_tensor('abc xyz')

### Generador de lotes (batches) 

La mayoría de las veces en el Procesamiento del Lenguaje Natural, y en la IA en general, usamos lotes (batches) al entrenar nuestros conjuntos de datos. Aquí, construiremos un generador de datos que reciba un texto y devuelva un lote de líneas de texto (las líneas son oraciones).
- El generador convierte las líneas de texto (oraciones) en arreglos de numpy de enteros rellenados con ceros para que todos los arreglos tengan la misma longitud, que es la longitud de la oración más larga en todo el conjunto de datos.

Una vez que creeremos el generador, puedes iterar sobre él así:

```
next(data_generator)
```

Este generador devuelve los datos en un formato que podremos usar directamente en nuestro modelo al calcular la propagación hacia adelante (feed-forward) del algoritmo. Este iterador devuelve un lote de líneas y una máscara por token. El lote es una tupla de tres partes: inputs, targets, mask. Los inputs y targets son idénticos. La segunda columna se usa para evaluar las predicciones. La máscara es 1 para los tokens que no son de relleno (padding).

**Implementaremos el generador de datos** 


In [ ]:
def data_generator(batch_size, max_length, data_lines, line_to_tensor=line_to_tensor, shuffle=True):
    # inicializar el indice que apunta a la posicion actual en el arreglo de indices de lineas
    index = 0
    
    # inicializar la lista que contendra el lote actual
    cur_batch = []
    
    # contar el numero de lineas en data_lines
    num_lines = len(data_lines)
    
    # crear un arreglo con los indices de data_lines que pueden mezclarse
    lines_index = [*range(num_lines)]
    
    # mezclar los indices de lineas si shuffle es True
    if shuffle:
        rnd.shuffle(lines_index)
    
    ### START CODE HERE (Replace instances of 'None' with your code) ###
    while True:
        
        # si el indice es mayor o igual al numero de lineas en data_lines
        if index >= num_lines:
            # entonces reiniciar el indice a 0
            index = 0
            # mezclar los indices de lineas si shuffle es True
            if shuffle:
                rnd.shuffle(lines_index)
            
        # obtener una linea en la posicion `lines_index[index]` en data_lines
        line = data_lines[lines_index[index]]
        
        # si la longitud de la linea es menor que max_length
        if len(line) < max_length:
            # agregar la linea al lote actual
            cur_batch.append(line)
            
        # incrementar el indice en uno
        index += 1
        
        # si el lote actual ahora es igual al tamano de lote deseado
        if len(cur_batch) == batch_size:
            
            batch = []
            mask = []
            
            # recorrer cada linea (li) en cur_batch
            for li in cur_batch:
                # convertir la linea (li) a un tensor de enteros
                tensor = line_to_tensor(li)
                
                # Crear una lista de ceros para representar el relleno (padding)
                # de modo que el tensor mas el relleno tenga longitud `max_length`
                pad = [0] * (max_length - len(tensor))
                
                # combinar el tensor mas el relleno
                tensor_pad = tensor + pad
                
                # agregar el tensor rellenado al lote
                batch.append(tensor_pad)

                # Una mascara para tensor_pad es 1 dondequiera que tensor_pad no sea
                # 0 y 0 dondequiera que tensor_pad sea 0, es decir, si tensor_pad es
                # [1, 2, 3, 0, 0, 0] entonces example_mask debe ser
                # [1, 1, 1, 0, 0, 0]
                # Pista: Usa una comprension de listas para esto
                example_mask = [0 if t == 0 else 1 for t in tensor_pad]
                mask.append(example_mask)
               
            # convertir el lote (tipo de dato lista) a un arreglo numpy de trax
            batch_np_arr = np.array(batch)
            mask_np_arr = np.array(mask)
            
            ### END CODE HERE ##
            
            # Producir (yield) dos copias del lote y la mascara.
            yield batch_np_arr, batch_np_arr, mask_np_arr
            
            # reiniciar el lote actual a una lista vacia
            cur_batch = []
            

In [ ]:
# Prueba tu generador de datos
tmp_lines = ['12345678901', # longitud 11
             '123456789', # longitud 9
             '234567890', # longitud 9
             '345678901'] # longitud 9

# Obtener un tamano de lote de 2, longitud maxima 10
tmp_data_gen = data_generator(batch_size=2, 
                              max_length=10, 
                              data_lines=tmp_lines,
                              shuffle=False)

# obtener un lote
tmp_batch = next(tmp_data_gen)

# ver el lote
tmp_batch

Ahora que tienes tu generador, puedes simplemente llamarlo y devolverá tensores que corresponden a tus líneas de Shakespeare. La primera columna y la segunda columna son idénticas. Ahora puedes continuar y comenzar a construir tu red neuronal. 

### Generador de lotes repetido 

De la forma en que está definido actualmente el iterador, seguirá proporcionando lotes para siempre.

Usualmente queremos recorrer el conjunto de datos varias veces durante el entrenamiento (es decir, entrenar durante múltiples *épocas*).

Para conjuntos de datos pequeños podemos usar [`itertools.cycle`](https://docs.python.org/3.8/library/itertools.html#itertools.cycle) para lograr esto fácilmente.

In [ ]:
import itertools
infinite_data_generator = itertools.cycle(
    data_generator(batch_size=2, max_length=10, data_lines=tmp_lines))

Puedes ver que podemos obtener más de las 5 líneas en tmp_lines usando esto.

In [ ]:
ten_lines = [next(infinite_data_generator) for _ in range(10)]
print(len(ten_lines))

# Definiendo el modelo GRU

Ahora que tenemos los tensores de entrada y salida, continuaremos e inicializaremos el modelo. Implementaremos el `GRULM`, el modelo de unidad recurrente con compuertas (gated recurrent unit). Para implementar este modelo, usaremos el paquete `trax` de google. En lugar de implementar la `GRU` desde cero, tenemos algunos métodos necesarios de un paquete incorporado. Podemos usar los siguientes paquetes al construir el modelo: 

- `tl.Serial`: Combinador que aplica capas en serie (por composición de funciones). [docs](https://trax-ml.readthedocs.io/en/latest/trax.layers.html#trax.layers.combinators.Serial) / [código fuente](https://github.com/google/trax/blob/1372b903bb66b0daccee19fd0b1fdf44f659330b/trax/layers/combinators.py#L26)
    - Puedes pasar las capas como argumentos a `Serial`, separadas por comas. 
    - Por ejemplo: `tl.Serial(tl.Embeddings(...), tl.Mean(...), tl.Dense(...), tl.LogSoftmax(...))`

___

- `tl.ShiftRight`: Permite que el modelo vaya hacia la derecha en la propagación hacia adelante (feed forward). [docs](https://trax-ml.readthedocs.io/en/latest/trax.layers.html#trax.layers.attention.ShiftRight) / [código fuente](https://github.com/google/trax/blob/1372b903bb66b0daccee19fd0b1fdf44f659330b/trax/layers/attention.py#L297)
    - Capa `ShiftRight(n_shifts=1, mode='train')` para desplazar el tensor hacia la derecha n_shift veces
    - Aquí en el ejercicio solo necesitas especificar el mode y no preocuparte por n_shifts

___

- `tl.Embedding`: Inicializa el embedding. En este caso es el tamaño del vocabulario por la dimensión del modelo. [docs](https://trax-ml.readthedocs.io/en/latest/trax.layers.html#trax.layers.core.Embedding) / [código fuente](https://github.com/google/trax/blob/1372b903bb66b0daccee19fd0b1fdf44f659330b/trax/layers/core.py#L113) 
    - `tl.Embedding(vocab_size, d_feature)`.
    - `vocab_size` es el número de palabras únicas en el vocabulario dado.
    - `d_feature` es el número de elementos en el embedding de palabras (algunas opciones para un tamaño de embedding de palabras van de 150 a 300, por ejemplo).
___

- `tl.GRU`: Capa GRU de `Trax`. [docs](https://trax-ml.readthedocs.io/en/latest/trax.layers.html#trax.layers.rnn.GRU) / [código fuente](https://github.com/google/trax/blob/1372b903bb66b0daccee19fd0b1fdf44f659330b/trax/layers/rnn.py#L143)
    - `GRU(n_units)` Construye una GRU tradicional de n_cells con transformaciones internas densas.
    - Artículo de `GRU`: https://arxiv.org/abs/1412.3555
___

- `tl.Dense`: Una capa densa. [docs](https://trax-ml.readthedocs.io/en/latest/trax.layers.html#trax.layers.core.Dense) / [código fuente](https://github.com/google/trax/blob/1372b903bb66b0daccee19fd0b1fdf44f659330b/trax/layers/core.py#L28)
    - `tl.Dense(n_units)`: El parámetro `n_units` es el número de unidades elegidas para esta capa densa.
___

- `tl.LogSoftmax`: Logaritmo de las probabilidades de salida. [docs](https://trax-ml.readthedocs.io/en/latest/trax.layers.html#trax.layers.core.LogSoftmax) / [código fuente](https://github.com/google/trax/blob/1372b903bb66b0daccee19fd0b1fdf44f659330b/trax/layers/core.py#L242)
    - Aquí, no necesitas establecer ningún parámetro para `LogSoftMax()`.
___

**Implementaremos la clase `GRULM` a continuación.**

In [ ]:
def GRULM(vocab_size=256, d_model=512, n_layers=2, mode='train'):
    model = tl.Serial(
      tl.ShiftRight(mode=mode), # Apilar la capa ShiftRight
      tl.Embedding(vocab_size=vocab_size, d_feature=d_model), # Apilar la capa de embedding
      [tl.GRU(n_units=d_model) for _ in range(n_layers)], # Apilar capas GRU de d_model unidades teniendo en cuenta el parametro n_layer (usa la sintaxis de comprension de listas)
      tl.Dense(n_units=vocab_size), # Capa densa
      tl.LogSoftmax() # Log Softmax
    )

    return model


In [ ]:
# probando tu modelo
model = GRULM()
print(model)


# Entrenamiento

Ahora entrenaremos el modelo. Como de costumbre, definiremos la función de costo, el optimizador, y decidir si entrenaremos en una `gpu` o `cpu`. También tenemos que alimentar un modelo ya construido. Antes de entrar al entrenamiento, utilizaremos las abstracciones `TrainTask` y `EvalTask`.

Para entrenar un modelo, Trax define una abstracción `trax.supervised.training.TrainTask` que empaqueta los datos de entrenamiento, la pérdida (loss) y el optimizador (entre otras cosas) en un objeto.

De manera similar, para evaluar un modelo, Trax define una abstracción `trax.supervised.training.EvalTask` que empaqueta los datos de evaluación y las métricas (entre otras cosas) en otro objeto.

La pieza final que une todo es la abstracción `trax.supervised.training.Loop` que es una forma muy simple y flexible de juntar todo y entrenar el modelo, todo mientras lo evalúa y guarda puntos de control (checkpoints).
Usar `training.Loop` nos ahorrará mucho código en comparación con escribir siempre el bucle de entrenamiento a mano. Más importante aún, es menos probable que tengamos un error en ese código que arruinaría tu entrenamiento.

In [ ]:
batch_size = 32
max_length = 64

Una `época` (epoch) se define tradicionalmente como una pasada a través del conjunto de datos.

Dado que el conjunto de datos se dividió en `lotes` (batches), necesitas varios `pasos` (steps, evaluaciones del gradiente) para completar una `época`. Así que, una `época` corresponde al número de ejemplos en un `lote` por el número de `pasos`. En resumen, en cada `época` se recorre todo el conjunto de datos. 

La variable `max_length` define la longitud máxima de las líneas que se usarán para entrenar nuestros datos; las líneas más largas que esa longitud se descartan. 

A continuación se muestra una función y resultados que indican cuántas líneas cumplen con nuestro criterio de longitud máxima de una oración en todo el conjunto de datos y cuántos `pasos` se requieren para cubrir todo el conjunto de datos, lo que a su vez corresponde a una `época`.

In [ ]:
def n_used_lines(lines, max_length):
    
    n_lines = 0
    for l in lines:
        if len(l) <= max_length:
            n_lines += 1
    return n_lines



In [ ]:
num_used_lines = n_used_lines(lines, 32)
print('Número de líneas utilizadas del conjunto de datos:', num_used_lines)
print('Tamaño del lote (una potencia de 2):', int(batch_size))
steps_per_epoch = int(num_used_lines/batch_size)
print('Número de pasos para completar una época:', steps_per_epoch)

### Entrenando el modelo

Ahora escribiremos una función que reciba el modelo y lo entrene. Para entrenar el modelo tenemos que decidir cuántas veces queremos iterar sobre todo el conjunto de datos. 


**Implementaremos la función `train_model`** para entrenar la red neuronal. Aquí hay una lista de cosas que debemos considerar:

- Crea un objeto `trax.supervised.trainer.TrainTask`, esto encapsula los aspectos del conjunto de datos y el problema en cuestión:
    - labeled_data = los datos etiquetados sobre los que queremos *entrenar*.
    - loss_fn = [tl.CrossEntropyLoss()](https://trax-ml.readthedocs.io/en/latest/trax.layers.html?highlight=CrossEntropyLoss#trax.layers.metrics.CrossEntropyLoss)
    - optimizer = [trax.optimizers.Adam()](https://trax-ml.readthedocs.io/en/latest/trax.optimizers.html?highlight=Adam#trax.optimizers.adam.Adam) con tasa de aprendizaje (learning rate) = 0.0005

- Crearemos un objeto `trax.supervised.trainer.EvalTask`, esto encapsula aspectos de la evaluación del modelo:
    - labeled_data = los datos etiquetados sobre los que queremos *evaluar*.
    - metrics = [tl.CrossEntropyLoss()](https://trax-ml.readthedocs.io/en/latest/trax.layers.html#trax.layers.metrics.CrossEntropyLoss) y [tl.Accuracy()](https://trax-ml.readthedocs.io/en/latest/trax.layers.html#trax.layers.metrics.Accuracy)
    - Con qué frecuencia queremos evaluar y guardar el punto de control (checkpoint) del modelo.

- Crearemos un objeto `trax.supervised.trainer.Loop`, esto encapsula lo siguiente:
    - Los objetos `TrainTask` y `EvalTask` creados anteriormente.
    - el modelo de entrenamiento = [GRULM](#ex03)
    - opcionalmente el modelo de evaluación, si es diferente del modelo de entrenamiento. NOTA: en presencia de Dropout, etc., usualmente queremos que el modelo de evaluación se comporte de forma ligeramente diferente al modelo de entrenamiento.

Usaremos una pérdida de entropía cruzada (cross entropy loss), con el optimizador Adam. Por favor lee la documentación de [trax](https://trax-ml.readthedocs.io/en/latest/index.html) para obtener una comprensión completa. 


In [ ]:
from trax.supervised import training

def train_model(model, data_generator, batch_size=32, max_length=64, lines=lines, eval_lines=eval_lines, n_steps=1, output_dir='Data/model'): 
    
    bare_train_generator = data_generator(batch_size, max_length, data_lines=lines)
    infinite_train_generator = itertools.cycle(bare_train_generator)
    
    bare_eval_generator = data_generator(batch_size, max_length, data_lines=eval_lines)
    infinite_eval_generator = itertools.cycle(bare_eval_generator)
   
    train_task = training.TrainTask(
        labeled_data=infinite_train_generator, # Usar el generador infinito de datos de entrenamiento
        loss_layer=tl.CrossEntropyLoss(),   # No olvides instanciar este objeto
        optimizer=trax.optimizers.Adam(0.0005)     # No olvides agregar el parametro de tasa de aprendizaje (learning rate)
    )

    eval_task = training.EvalTask(
        labeled_data=infinite_eval_generator,    # Usar el generador infinito de datos de evaluacion
        metrics=[tl.CrossEntropyLoss(), tl.Accuracy()], # No olvides instanciar estos objetos
        n_eval_batches=3      # Para mejor exactitud de evaluacion en un tiempo razonable
    )
    
    training_loop = training.Loop(model,
                                  train_task,
                                  eval_tasks=[eval_task],
                                  output_dir=output_dir)

    training_loop.run(n_steps=n_steps)
        
    # Devolvemos esto porque contiene una referencia al modelo, que tiene los pesos, etc.
    return training_loop


In [ ]:
# Entrenar el modelo 1 paso y conservar el objeto `trax.supervised.training.Loop`.
training_loop = train_model(GRULM(), data_generator)

El modelo solo se entrenó durante 1 paso debido a las limitaciones de este entorno. Incluso en un entorno acelerado por GPU, le tomaría muchas horas alcanzar un buen nivel de exactitud. Para el resto de este archivo de jupyter usaremos un modelo preentrenado.

# Evaluación  

### Evaluando usando las redes profundas

Ahora que sabemos cómo entrenar un modelo, tendremos ver como evaluarlo. Para evaluar modelos de lenguaje, usualmente usamos la perplejidad, que es una medida de qué tan bien un modelo de probabilidad predice una muestra. Observa que la perplejidad se define como: 

$$P(W) = \sqrt[N]{\prod_{i=1}^{N} \frac{1}{P(w_i| w_1,...,w_{n-1})}}$$

Como truco de implementación, usualmente tomarías el logaritmo de esa fórmula (para permitirnos usar las log-probabilidades que obtenemos como salida de nuestra `RNN`, convertir exponentes en productos, y productos en sumas, lo que hace que los cálculos sean menos complicados y computacionalmente más eficientes). También debemos tener cuidado con el relleno (padding), ya que no queremos incluir el padding al calcular la perplejidad (porque no queremos tener una medida de perplejidad artificialmente buena).


$$log P(W) = {log\big(\sqrt[N]{\prod_{i=1}^{N} \frac{1}{P(w_i| w_1,...,w_{n-1})}}\big)}$$

$$ = {log\big({\prod_{i=1}^{N} \frac{1}{P(w_i| w_1,...,w_{n-1})}}\big)^{\frac{1}{N}}}$$ 

$$ = {log\big({\prod_{i=1}^{N}{P(w_i| w_1,...,w_{n-1})}}\big)^{-\frac{1}{N}}} $$
$$ = -\frac{1}{N}{log\big({\prod_{i=1}^{N}{P(w_i| w_1,...,w_{n-1})}}\big)} $$
$$ = -\frac{1}{N}{\big({\sum_{i=1}^{N}{logP(w_i| w_1,...,w_{n-1})}}\big)} $$


**Escribiremos una función que ayude a evaluar el modelo.** La funcón recibirá preds y target. Preds es un tensor de log-probabilidades. Podemos usar [`tl.one_hot`](https://github.com/google/trax/blob/22765bb18608d376d8cd660f9865760e4ff489cd/trax/layers/metrics.py#L154) para transformar el target en la misma dimensión. Luego los multiplicaremos y sumaremos. 

También crearemos una máscara para obtener solo las probabilidades sin relleno (non-padded).

In [ ]:
def test_model(preds, target):
   
    total_log_ppx = np.sum(preds * tl.one_hot(target, preds.shape[-1]),axis= -1)

    non_pad = 1.0 - np.equal(target, 0)          # Debes comprobar si el target es igual a 0
    ppx = total_log_ppx * non_pad                       # Eliminar el relleno (padding)

    log_ppx = np.sum(ppx) / np.sum(non_pad)

    
    return -log_ppx

In [ ]:
model = GRULM()
model.init_from_file('Data/model.pkl.gz')
batch = next(data_generator(batch_size, max_length, lines, shuffle=False))
preds = model(batch[0])
log_ppx = test_model(preds, batch[1])
print('La perplejidad logarítmica y la perplejidad de su modelo son, respectivamente', log_ppx, np.exp(log_ppx))

# Generando el lenguaje con el modelo

Ahora usaremos el modelo de lenguaje para generar nuevas oraciones, para eso necesitamos hacer extracciones de una distribución de Gumbel.

La Función de Densidad de Probabilidad (PDF) de Gumbel se define como: 

$$ f(z) = {1\over{\beta}}e^{(-z+e^{(-z)})} $$

donde: $$ z = {(x - \mu)\over{\beta}}$$

El valor máximo, que es lo que elegimos como la predicción en el último paso de una Red Neuronal Recursiva `RNN` que estamos usando para la generación de texto, en una muestra de una variable aleatoria que sigue una distribución exponencial se aproxima a la distribución de Gumbel cuando la muestra aumenta asintóticamente. Por esa razón, la distribución de Gumbel se usa para muestrear de una distribución categórica.

In [ ]:
# Ejecuta esta celda para generar alguna oracion nueva
def gumbel_sample(log_probs, temperature=1.0):
    """Muestreo de Gumbel de una distribucion categorica."""
    u = np.random.uniform(low=1e-6, high=1.0 - 1e-6, size=log_probs.shape)
    g = -np.log(-np.log(u))
    return np.argmax(log_probs + g * temperature, axis=-1)

def predict(num_chars, prefix):
    inp = [ord(c) for c in prefix]
    result = [c for c in prefix]
    max_len = len(prefix) + num_chars
    for _ in range(num_chars):
        cur_inp = np.array(inp + [0] * (max_len - len(inp)))
        outp = model(cur_inp[None, :])  # Agregar dimension de lote (batch).
        next_char = gumbel_sample(outp[0, len(inp)])
        inp += [int(next_char)]
       
        if inp[-1] == 1:
            break  # EOS (fin de oracion)
        result.append(chr(int(next_char)))
    
    return "".join(result)

print(predict(32, ""))

In [ ]:
print(predict(32, ""))
print(predict(32, ""))
print(predict(32, ""))


En el texto generado arriba, se puede ver que el modelo genera texto que tiene sentido, capturando dependencias entre palabras y sin ninguna entrada. Un simple modelo de n-gramas no habría podido capturar todo eso en una oración.

###  Sobre los métodos estadísticos 

Usar un método estadístico como el que implementamos no nos dará resultados tan buenos. El modelo no podrá codificar información vista previamente en el conjunto de datos y, como resultado, la perplejidad aumentará. Recordemos que cuanto mayor es la perplejidad, peor es el modelo. Además, los modelos estadísticos de n-gramas ocupan demasiado espacio y memoria. Como resultado, serán ineficientes y demasiado lentos. Por el contrario, con las redes profundas (deepnets), podemos obtener una mejor perplejidad. 
